# Full Trenberth Diagram Calibration

This notebook calibrates **23 parameters** spanning shortwave radiation, longwave radiation, surface albedo, and land hydrology to simultaneously match the [Trenberth et al. (2009)](https://doi.org/10.1175/2008BAMS2634.1) global energy budget across **6 fluxes**:

| Flux | Target | Description |
|------|--------|-------------|
| OSR  | 101.9 W/m² | Outgoing shortwave at top-of-atmosphere |
| SRU  | 23.0 W/m²  | Surface shortwave reflected upward |
| SRD  | 168.0 W/m² | Surface shortwave absorbed downward |
| OLR  | 235.0 W/m² | Outgoing longwave at top-of-atmosphere |
| LRD  | 333.0 W/m² | Surface longwave downward (greenhouse back-radiation) |
| LRU  | 398.0 W/m² | Surface longwave upward (Stefan-Boltzmann emission) |

This is an extension of `radiation_sw.ipynb`: we add longwave emissivities and land-hydrology parameters. Including OLR in the loss couples the SW and LW optimisation — changing cloud albedo now also feeds back through the LW budget via surface temperature.

**Expected runtime:** ~8–12 hours at T31. Use Section 5 (quick test) first.

## 1. Setup

In [2]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

using SpeedyCalibration
using Optimisers
using CairoMakie
using GeoMakie
using Dates
using Printf

  Activating project at `~/master_thesis/Code_SpeedyWeather/SpeedyCalibration.jl`
Precompiling packages...
 155731.9 ms  ✓ SpeedyCalibration
  1 dependency successfully precompiled in 157 seconds. 214 already precompiled.
[ Info: Precompiling SpeedyCalibration [5d65bc14-e915-412c-9a7c-d2e552044b02]
[ Info: Precompiling CairoMakie [13f3f980-e62b-5c42-98c6-ff1f3baf88f0]
[ Info: Precompiling ModelParametersUnitfulExt [30a72cc9-359b-520d-a5bb-2064a7c97f0f]
[ Info: Precompiling ModelParametersMakieExt [fe55f67a-76af-556c-8c6e-f9ee716b975d]
[ Info: Precompiling DomainSetsMakieExt [da481366-01c8-5b4d-b359-47a10c8532e7]
[ Info: Precompiling RingGridsMakieExt [e5dccbbd-f882-5bb0-bb40-f730cea6c903]
[ Info: Precompiling GeoMakie [db073c08-6b98-4ee5-b6a4-5efafb3259c6]
[ Info: Precompiling RingGridsGeoMakieExt [2d315563-22ec-53cd-bb86-4034d5a6f7a6]
[ Info: Precompiling SpeedyWeatherGeoMakieExt [e8a06b79-132c-5f7f-a25c-66406c5f936d]
[ Info: Precompiling SpeedyCalibrationMakieExt [657b58ae-5d3d-567e-

## 2. Define Trainable Parameters

We train **23 parameters** across four modules. All paths and bounds are copied
directly from `full_trend_birth_training.ipynb`.

**`absorptivity_water_vapor` needs `grad_scale=0.01`.**
The physical gradient is ~200× weaker than the cloud parameters because it is
multiplied by specific humidity (q ≈ 0.005). Its lower bound is 60 (not 0) because
the model becomes numerically unstable below ~57.

**`snow_melting_threshold` needs `grad_scale=0.01`.**
Measured in Kelvin (~275 K), so a 1 W/m² flux change produces a ~100× smaller
raw gradient than dimensionless parameters.

**Parameters with zero gradient (excluded):**
- `conv_time_scale` — integer conversion in `Second(time_scale).value` is opaque to Enzyme
- `lsc_rh_threshold` — step-function boundary; gradient is zero almost everywhere


In [3]:
param_specs = [
    # ── SW cloud reflection ───────────────────────────────────────────────────
    ParamSpec(:cloud_albedo,
        [:shortwave_radiation, :clouds, :cloud_albedo];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),
    ParamSpec(:stratocumulus_cover_max,
        [:shortwave_radiation, :clouds, :stratocumulus_cover_max];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),
    ParamSpec(:stratocumulus_albedo,
        [:shortwave_radiation, :clouds, :stratocumulus_albedo];
        bounds=(0.10f0, 0.90f0), initial=0.50f0),
    ParamSpec(:precipitation_weight,
        [:shortwave_radiation, :clouds, :precipitation_weight];
        bounds=(0.0f0, 0.8f0), initial=0.20f0),

    # ── SW atmospheric absorption ─────────────────────────────────────────────
    ParamSpec(:absorptivity_water_vapor,
        [:shortwave_radiation, :transmissivity, :absorptivity_water_vapor];
        bounds=(60f0, 140f0), initial=75f0, grad_scale=0.01f0),
    ParamSpec(:absorptivity_dry_air,
        [:shortwave_radiation, :transmissivity, :absorptivity_dry_air];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),
    ParamSpec(:absorptivity_aerosol,
        [:shortwave_radiation, :transmissivity, :absorptivity_aerosol];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),
    ParamSpec(:ozone_absorption,
        [:shortwave_radiation, :radiative_transfer, :ozone_absorption];
        bounds=(0.002f0, 0.020f0), initial=0.01f0),

    # ── Surface albedo ────────────────────────────────────────────────────────
    ParamSpec(:albedo_land,
        [:albedo, :land, :albedo_land];
        bounds=(0.10f0, 0.70f0), initial=0.40f0),
    ParamSpec(:albedo_high_vegetation,
        [:albedo, :land, :albedo_high_vegetation];
        bounds=(0.04f0, 0.26f0), initial=0.15f0),
    ParamSpec(:albedo_low_vegetation,
        [:albedo, :land, :albedo_low_vegetation];
        bounds=(0.05f0, 0.35f0), initial=0.20f0),
    ParamSpec(:albedo_snow,
        [:albedo, :land, :albedo_snow];
        bounds=(0.15f0, 0.75f0), initial=0.40f0),
    ParamSpec(:snow_depth_scale,
        [:albedo, :land, :snow_depth_scale];
        bounds=(0.005f0, 0.20f0), initial=0.05f0),
    ParamSpec(:albedo_ocean,
        [:albedo, :ocean, :albedo_ocean];
        bounds=(0.02f0, 0.10f0), initial=0.06f0),
    ParamSpec(:albedo_ice,
        [:albedo, :ocean, :albedo_ice];
        bounds=(0.30f0, 0.90f0), initial=0.60f0),

    # ── Longwave transmissivity (Frierson scheme) ─────────────────────────────
    # τ₀_equator and τ₀_pole control optical depth; fₗ is the LW fraction.
    ParamSpec(:tau0_equator,
        [:longwave_radiation, :transmissivity, :τ₀_equator];
        bounds=(2f0, 12f0), initial=6f0),
    ParamSpec(:tau0_pole,
        [:longwave_radiation, :transmissivity, :τ₀_pole];
        bounds=(0.3f0, 4f0), initial=1.5f0),
    ParamSpec(:fl,
        [:longwave_radiation, :transmissivity, :fₗ];
        bounds=(0.0f0, 0.5f0), initial=0.1f0),

    # ── Longwave emissivity ───────────────────────────────────────────────────
    ParamSpec(:emissivity_ocean,
        [:longwave_radiation, :radiative_transfer, :emissivity_ocean];
        bounds=(0.80f0, 1.00f0), initial=0.98f0),
    ParamSpec(:emissivity_land,
        [:longwave_radiation, :radiative_transfer, :emissivity_land];
        bounds=(0.80f0, 1.00f0), initial=0.98f0),

    # ── Land hydrology ────────────────────────────────────────────────────────
    ParamSpec(:infiltration_fraction,
        [:land, :soil_moisture, :infiltration_fraction];
        bounds=(0.05f0, 0.80f0), initial=0.25f0),
    ParamSpec(:ocean_moisture,
        [:land, :soil_moisture, :ocean_moisture];
        bounds=(0.0f0, 1.0f0), initial=0.0f0),
    ParamSpec(:snow_melting_threshold,
        [:land, :snow, :melting_threshold];
        bounds=(270f0, 280f0), initial=275f0, grad_scale=0.01f0),
]

println("$(length(param_specs)) trainable parameters defined.")


23 trainable parameters defined.


## 3. Configure the Loss Function

`TRENBERTH_LOSS` targets all 6 SW+LW fluxes. OLR and OSR receive full weight (1.0); surface fluxes receive lower weight (0.3–0.5) reflecting their larger observational uncertainty.

In [4]:
loss_config = TRENBERTH_LOSS

println("Loss configuration: $(length(loss_config.flux_keys))-flux MSE")
println()
@printf("  %-6s  %8s  %6s\n", "flux", "target", "weight")
println("  " * "-" ^ 24)
for k in loss_config.flux_keys
    @printf("  %-6s  %6.1f W/m²  %.1f\n",
            k, loss_config.targets[k], loss_config.weights[k])
end

Loss configuration: 6-flux MSE

  flux      target  weight
  ------------------------
  osr      101.9 W/m²  1.0
  sru       23.0 W/m²  0.5
  srd      168.0 W/m²  0.5
  olr      235.0 W/m²  1.0
  lrd      333.0 W/m²  0.3
  lru      398.0 W/m²  0.3


## 4. Quick Test

Verify all parameter paths are correct and Enzyme can differentiate through everything before the full run.

> **Note:** `calibrate!` warms up Enzyme automatically on the actual training model (`warmup_enzyme=true` by default). Expect *"Enzyme warmup complete in X s."* before the spinup on the first call. Pass `warmup_enzyme=false` on subsequent calls in the same session.

In [ ]:
result_test = calibrate!(
    param_specs,
    Optimisers.Adam(1f-2),
    loss_config,
    quick_test_config(),
)

println("Quick test complete: ", result_test.conv_info.stop_reason)

# Check gradient magnitudes — a zero gradient for any parameter is a red flag
println("\nGradient magnitudes (last batch):")
@printf("  %-28s  %12s\n", "parameter", "|mean grad|")
println("  " * "-" ^ 45)
for spec in result_test.param_specs
    g = result_test.history[Symbol("grad_", spec.name)]
    isempty(g) && continue
    @printf("  %-28s  %12.3e\n", spec.name, abs(g[end]))
end

[ Info: Time step changed from 12800000 to 10800000 milliseconds (-16%) to match output frequency.
[ Info: Time step changed from 12800000 to 10800000 milliseconds (-16%) to match output frequency.


SpeedyCalibration.jl — calibrate!
   1. cloud_albedo                     = 0.6000  [0.250, 0.950]  raw₀=0.000
   2. stratocumulus_cover_max          = 0.6000  [0.250, 0.950]  raw₀=0.000
   3. stratocumulus_albedo             = 0.5000  [0.100, 0.900]  raw₀=0.000
   4. precipitation_weight             = 0.2000  [0.000, 0.800]  raw₀=-1.099
   5. absorptivity_water_vapor         = 75.0000  [60.000, 140.000]  raw₀=-1.466  [×0]
   6. absorptivity_dry_air             = 0.0314  [0.005, 0.060]  raw₀=-0.084
   7. absorptivity_aerosol             = 0.0314  [0.005, 0.060]  raw₀=-0.084
   8. ozone_absorption                 = 0.0100  [0.002, 0.020]  raw₀=-0.223
   9. albedo_land                      = 0.4000  [0.100, 0.700]  raw₀=0.000
  10. albedo_high_vegetation           = 0.1500  [0.040, 0.260]  raw₀=0.000
  11. albedo_low_vegetation            = 0.2000  [0.050, 0.350]  raw₀=0.000
  12. albedo_snow                      = 0.4000  [0.150, 0.750]  raw₀=-0.336
  13. snow_depth_scale                

**Interpreting gradient magnitudes:**  
- All gradients should be non-zero. A zero gradient means the parameter is disconnected from the loss through the AD graph — check the path or exclude the parameter.
- If one gradient is 100× larger than the others, consider increasing `grad_scale` for the weaker ones, or reducing it for the outlier.

---

## 5. Full Training Run

**⏱ Expected runtime: ~8–12 hours at T31.**

We use a slightly lower initial LR (`5f-3`) compared to the SW-only run because the 6-flux loss is more sensitive and can oscillate with a high LR.

In [ ]:
result = calibrate!(
    param_specs,
    Optimisers.Adam(5f-3),
    loss_config,
    TrainingConfig(
        spinup_days       = 180,
        batch_days        = 3.0,
        samples_per_batch = 20,
        max_batches       = 300,
        loss_threshold    = 500f0,
        trunc             = 31,
        nlayers           = 8,
    ),
)

save_path = joinpath(@__DIR__, "output", "trenberth_full_result.jld2")
mkpath(dirname(save_path))
save_result(result, save_path)
println("Result saved to: $save_path")

## 6. Inspect Results

In [ ]:
println(result)
println()
println("Convergence info:")
println("  stop_reason:        ", result.conv_info.stop_reason)
println("  best_smoothed_loss: ", round(result.conv_info.best_smoothed_loss, digits=2))
println("  total_batches:      ", result.conv_info.total_batches)
@printf("  total_time:         %.1f hours\n", result.conv_info.total_time / 3600)

In [ ]:
# Parameter table: initial → trained
@printf("\n%-28s  %10s  %10s  %10s\n", "parameter", "initial", "trained", "change")
println("-" ^ 65)
for spec in result.param_specs
    init    = isnothing(spec.initial) ? NaN32 : spec.initial
    trained = result.final_params[spec.name]
    @printf("%-28s  %10.4g  %10.4g  %+10.4g\n",
            spec.name, init, trained, trained - init)
end

In [ ]:
# Flux bias at end of training
@printf("\n%-6s  %8s  %8s  %8s\n", "flux", "target", "trained", "bias")
println("-" ^ 38)
for k in result.loss_config.flux_keys
    tgt  = result.loss_config.targets[k]
    val  = result.history[k][end]
    @printf("%-6s  %8.2f  %8.2f  %+8.2f\n", k, tgt, val, val - tgt)
end

## 7. Plot Training History

In [ ]:
figs = plot_training(result; save_dir=joinpath(@__DIR__, "output", "trenberth_full"))
figs.fig_loss

In [ ]:
figs.fig_flux

In [ ]:
figs.fig_params

In [ ]:
figs.fig_grads

## 8. Climate Validation

We compare the default model's equilibrium climatology against the trained one across all 6 target fluxes, precipitation, and the temperature profile.

**⏱ Expected runtime: ~1–3 hours at T31.**

In [ ]:
clm = run_climate_validation(result; n_years=7, stat_years=5)

In [ ]:
# Full bias comparison table
targets = result.loss_config.targets

@printf("%-6s  %8s  %9s  %9s  %9s  %9s\n",
        "flux", "target",
        "def val", "def bias",
        "trn val", "trn bias")
println("-" ^ 60)
for k in result.loss_config.flux_keys
    tgt    = targets[k]
    d_val  = getproperty(clm.default, k)
    t_val  = getproperty(clm.trained, k)
    @printf("%-6s  %8.2f  %9.2f  %+9.2f  %9.2f  %+9.2f\n",
            k, tgt, d_val, d_val-tgt, t_val, t_val-tgt)
end
println()

# Also show precipitation (not in the loss — a held-out diagnostic)
println("Held-out diagnostics (not in loss):")
@printf("  Precipitation:  default = %.2f mm/day  trained = %.2f mm/day  (ERA5 ≈ 2.74)\n",
        clm.default.precip_total, clm.trained.precip_total)

In [ ]:
cfigs = plot_climate(clm;
    save_dir    = joinpath(@__DIR__, "output", "trenberth_full"),
    loss_config = result.loss_config,
)
cfigs.fig_rad

In [ ]:
cfigs.fig_lw

In [ ]:
cfigs.fig_precip

In [ ]:
cfigs.fig_summary

## 9. Interpretation and Next Steps

**Reading the summary plot:**  
Each bar shows the equilibrium bias (trained value − Trenberth target) for one flux. Bars shrinking toward zero from default (grey) to trained (blue) indicate successful calibration. Watch for biases that *grow* in fluxes not in the loss — these indicate compensatory parameter adjustments.

**Common issues and fixes:**

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| Loss oscillates without decreasing | LR too high | Reduce Adam LR to `1f-3` |
| One flux improves, another degrades | Loss weights too uneven | Rebalance `TRENBERTH_LOSS` weights |
| Parameter hits its bound | Bounds too narrow | Widen the offending `ParamSpec` bounds |
| Precipitation degrades strongly | Convective params pulled too far | Add precipitation term to loss |
| Gradient ~0 for a parameter | Zero-gradient path | Exclude parameter or change its path |

**Continuing from this result:**  
Set `initial = result.final_params[spec.name]` in each `ParamSpec` and re-run with a lower LR to refine further. The sigmoid reparameterisation ensures bounds are still respected.